# Task 1: Time-Series Preprocessing and Exploratory Analysis

**Dataset:** [Energy consumption, generation, prices and weather](https://www.kaggle.com/datasets/nicholasjhana/energy-consumption-generation-prices-and-weather) (Kaggle), hourly data for Spain from 2015 to 2018.
**Target:** `total load actual`, the hourly electricity demand in MW.

## Problem and dataset
The goal is to forecast Spain's hourly electricity demand from its own recent history and from the weather. Grid operators have to match supply to demand every hour, so forecast errors are costly.

I chose this dataset because it has a clear hourly timestamp, a meaningful numeric target, and many other variables measured over time (generation by source, prices, and weather for five cities). Demand also has daily, weekly and yearly patterns, which makes it a good fit for lagged features and moving averages.

The data comes as two files: `energy_dataset.csv` (one row per hour) and `weather_features.csv` (one row per city per hour).

In [3]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.patches import Patch
from IPython.display import display

# colours and plot style used throughout
BLUE, ORANGE, GRAY = "#2a78d6", "#eb6834", "#8a8985"
BLUES = LinearSegmentedColormap.from_list("blues", ["#e8f0fb", "#2a78d6", "#0d3a73"])
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "axes.titlesize": 12,
    "axes.titleweight": "bold", "axes.titlelocation": "left", "legend.frameon": False,
})

DATA_DIR = Path("data") if Path("data").exists() else Path(".")
OUT_DIR = Path("outputs")
FIG_DIR = OUT_DIR / "figures"
for folder in [FIG_DIR, OUT_DIR / "processed", OUT_DIR / "tables"]:
    folder.mkdir(parents=True, exist_ok=True)


def save(fig, name):
    fig.savefig(FIG_DIR / f"{name}.png", dpi=150, bbox_inches="tight")

## Preprocessing functions
The cleaning steps are written as small functions so the same pipeline can be applied again later. I explain each step in the sections below, where it is first used.

In [4]:
TARGET = "total load actual"
CITIES = ["Barcelona", "Bilbao", "Madrid", "Seville", "Valencia"]
WEATHER_VARS = ["temp", "humidity", "pressure", "wind_speed", "clouds_all", "rain_1h"]
LOCAL_TZ = "Europe/Madrid"

# energy columns that are completely empty or always 0
EMPTY_OR_ZERO_COLS = [
    "generation fossil coal-derived gas", "generation fossil oil shale",
    "generation fossil peat", "generation geothermal",
    "generation hydro pumped storage aggregated", "generation marine",
    "generation wind offshore", "forecast wind offshore eday ahead",
]

# generation columns where a 0 is a reporting dropout, not real output
# (each has only 1-4 zero hours in four years)
ZERO_IS_DROPOUT_COLS = [
    "generation biomass", "generation fossil gas", "generation fossil hard coal",
    "generation fossil oil", "generation hydro run-of-river and poundage",
    "generation hydro water reservoir", "generation nuclear", "generation other",
    "generation other renewable", "generation solar", "generation waste",
    "generation wind onshore",
]


def clean_energy(raw):
    """Parse time to UTC, drop useless columns, treat dropout zeros as missing, interpolate gaps."""
    df = raw.copy()
    df["time"] = pd.to_datetime(df["time"], utc=True)
    df = df.sort_values("time", kind="stable").drop_duplicates("time").set_index("time")
    df = df.drop(columns=EMPTY_OR_ZERO_COLS)
    df[ZERO_IS_DROPOUT_COLS] = df[ZERO_IS_DROPOUT_COLS].mask(df[ZERO_IS_DROPOUT_COLS] == 0)
    return df.interpolate(method="time", limit_direction="both")


def clean_weather(raw):
    """Remove duplicates and impossible values, convert to Celsius, one row per hour."""
    w = raw.copy()
    w["city_name"] = w["city_name"].str.strip()
    w["dt_iso"] = pd.to_datetime(w["dt_iso"], utc=True)
    w = w.sort_values(["city_name", "dt_iso"], kind="stable")
    w = w.drop_duplicates(["city_name", "dt_iso"], keep="first")

    w.loc[~w["pressure"].between(900, 1100), "pressure"] = np.nan   # hPa
    w.loc[w["wind_speed"] > 50, "wind_speed"] = np.nan              # m/s
    w["temp"] = w["temp"] - 273.15                                  # Kelvin to Celsius

    wide = []
    for city, g in w.groupby("city_name"):
        g = g.set_index("dt_iso")[WEATHER_VARS].interpolate(method="time", limit_direction="both")
        g.columns = [f"{c}_{city}" for c in WEATHER_VARS]
        wide.append(g)
    wide = pd.concat(wide, axis=1)
    for c in WEATHER_VARS:                                          # average over the five cities
        wide[f"{c}_mean"] = wide[[f"{c}_{city}" for city in CITIES]].mean(axis=1)
    wide.index.name = "time"
    return wide


def merge_energy_weather(energy, weather):
    """Join on the hourly UTC timestamp."""
    return energy.join(weather, how="left")


def add_time_features(df, tz=LOCAL_TZ):
    """Calendar features in local time, since demand follows the local clock."""
    df = df.copy()
    local = df.index.tz_convert(tz)
    df["hour"] = local.hour
    df["dayofweek"] = local.dayofweek          # 0 = Monday
    df["month"] = local.month
    df["year"] = local.year
    df["is_weekend"] = (local.dayofweek >= 5).astype(int)
    return df


def add_lag_features(df, target=TARGET, lags=(1, 24, 168), windows=(24, 168)):
    """Lagged load and moving averages. The averages use shift(1) so a row only
    uses information up to the previous hour."""
    df = df.copy()
    for lag in lags:
        df[f"load_lag_{lag}"] = df[target].shift(lag)
    for win in windows:
        df[f"load_ma_{win}"] = df[target].shift(1).rolling(win).mean()
    return df